# Listening Patterns

One person's listening, from Spotify's Extended Streaming History and the project's poller. The person is chosen when the report is built — `make report-02 PROFILE=<slug>` — so the same notebook serves anyone in the family who has provided an export, and the report's filename carries the slug. Coverage limits come first (§1), because every number after them inherits them. The notebook reads the warehouse only and makes no Spotify API call.

The project's notebook setup opens the warehouse for the profile named by `SPOT_PROFILE`, reads that profile's coverage window, and loads its plays once. Every panel below is computed from them.

In [ ]:
from spotify_lakehouse import listening
from spotify_lakehouse.notebook import setup

ctx = setup()
cov = listening.load_coverage(ctx)
plays = listening.load_plays(ctx)
listening.emit_intro(ctx, plays, cov)

## 1. Coverage & Caveats

Read this first. It states the window every chart is clipped to, how much of the listening carries a duration, and how much of the content is resolved to a Spotify object.

In [ ]:
listening.emit_coverage(ctx, plays, cov)

## 2. Volume Over Time

How much listening, and when. Listening time exists only in the export, so every series here is clipped to the export window, with any partial period at its edges shaded. Each chart's lower panel shows the plays behind it and the share carrying a duration.

### 2.1 Daily listening minutes

<!-- caption: Average minutes per day with a rolling mean; below, plays and the share carrying a duration -->

In [ ]:
listening.emit_daily_minutes(ctx, plays, cov)

### 2.2 Monthly totals

<!-- caption: Hours per calendar month; partial months at the window edges are hatched -->

In [ ]:
listening.emit_monthly_totals(ctx, plays, cov)

### 2.3 By day of week and daypart

<!-- caption: Hours by day of week and daypart, in the profile's home timezone -->

In [ ]:
listening.emit_dow_daypart(ctx, plays, cov)

### 2.4 Play count vs. qualified plays

A qualified play lasted at least 30 seconds or half the item's length (data-contracts §2); the gap between the two series is the listening that did not.

<!-- caption: Plays and qualified plays per period; below, the share that did not qualify -->

In [ ]:
listening.emit_qualified(ctx, plays, cov)

## 3. Music vs. Podcasts

Three categories, not two: besides music and podcasts, the export carries **audiobook chapters**, which are neither and are shown on their own.

### 3.1 Share of listening time

<!-- caption: Monthly share of listening time by category -->

In [ ]:
listening.emit_time_share(ctx, plays, cov)

### 3.2 Share of play count

<!-- caption: Monthly share of plays by category -->

In [ ]:
listening.emit_count_share(ctx, plays, cov)

### 3.3 Podcast leaderboard

The shows with the most listening time, with episodes played and plays behind each.

In [ ]:
listening.emit_podcast_leaderboard(ctx, plays, cov)

### 3.4 Podcast listening by daypart

<!-- caption: Podcast hours by daypart -->

In [ ]:
listening.emit_podcast_daypart(ctx, plays, cov)

## 4. Genre Spread

The allocation reconciliation comes first and stands on its own. The genre charts wait on the bucket mapping, and say so.

### 4.1 Allocation reconciliation

<!-- caption: Listening time kept at each allocation step, with what each step lost -->

In [ ]:
listening.emit_reconciliation(ctx, plays, cov)

### 4.2 Current spread

> [!CAUTION]
> **Not built: waits on R-015.** This radar needs the 13-bucket genre mapping, a business definition maintained as a spreadsheet seed (data-contracts §3). It does not exist yet, and raw genre strings are not a substitute for buckets.

### 4.3 Longitudinal

> [!CAUTION]
> **Not built: waits on R-015**, for the same reason as §4.2.

### 4.4 Drill-down

> [!CAUTION]
> **Not built: waits on R-015.** A drill-down from bucket to genre to artist has no first level without the mapping.

## 5. Artists & Tracks

Who and what, across the years.

### 5.1 Top artists by era

<!-- caption: Yearly rank of the artists that reached the top ten in more than one year -->

In [ ]:
listening.emit_top_artists(ctx, plays, cov)

### 5.2 Tracks with the longest tail

Tracks that have stayed in rotation longest.

In [ ]:
listening.emit_longest_tail(ctx, plays, cov)

### 5.3 One-hit wonders

Tracks played heavily for one month and never again.

In [ ]:
listening.emit_one_hit_wonders(ctx, plays, cov)

## 6. Behavior

How the listening happens: skips, shuffle, offline play, devices, and whether temporary relocations show in the data.

### 6.1 Skip rate over time

<!-- caption: Monthly skip rate; below, the plays behind it -->

In [ ]:
listening.emit_skip_rate(ctx, plays, cov)

### 6.2 Shuffle vs. deliberate

<!-- caption: Monthly share of plays on shuffle; below, the plays behind it -->

In [ ]:
listening.emit_shuffle(ctx, plays, cov)

### 6.3 Offline listening

<!-- caption: Monthly share of plays made offline -->

In [ ]:
listening.emit_offline(ctx, plays, cov)

### 6.4 Platform mix over time

<!-- caption: Yearly share of plays by device family -->

In [ ]:
listening.emit_platform(ctx, plays, cov)

### 6.5 Relocation signals — country and hour-of-day phase

Two independent signals for temporary relocations. The country a play connected from is a direct read. A shift in *when* listening happens is a measurement that needs interpreting, which is why this section names no place.

<!-- caption: Monthly shift of the mean listening hour against the surrounding months -->

In [ ]:
listening.emit_relocation(ctx, plays, cov)

The warehouse connection is closed at the end of the run.

In [ ]:
ctx.close()